In [16]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../../")

import nest_asyncio
nest_asyncio.apply()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import importlib.util
_real_find_spec = getattr(importlib.util, '_real_find_spec', importlib.util.find_spec)
importlib.util._real_find_spec = _real_find_spec
def _no_socksio(name, *a, **kw):
    if name == "socksio": return None
    return _real_find_spec(name, *a, **kw)
importlib.util.find_spec = _no_socksio

import datetime, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import psycopg2
import pytz

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.TimeseriesBuilder import TimeseriesBuilder
from TB.IRSwapsTB import IRSwapsTB
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue
from SDRUtils._swappulse_scripts.ingest_usdswaps_tape import resolve_pg_url

warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
NY = pytz.timezone("America/New_York")

_FLIP = {"PAID": "RECEIVED", "RECEIVED": "PAID"}


def _canonical_index(s):
    """Collapse any USD rate-index or curve name to a canonical token.

    Fed Funds appears on the tape and in curve/outright names under several
    conventions -- e.g. ``USD-Federal Funds-OIS Compound``,
    ``USD-Federal Funds-H.15-OIS-COMPOUND``, ``USD-Federal Funds-H.15``,
    ``USD-OIS-...-SERFFX-MIX23`` -- all the same funds leg for direction
    purposes. Returns ``"SOFR"`` / ``"FED_FUNDS"`` when recognisable, else the
    input unchanged (so tokens like ``BASIS``/``OTHER`` never match a filter).
    """
    if s is None or (isinstance(s, float) and s != s):
        return None
    u = str(s).upper()
    if "SOFR" in u:                       # check SOFR first (e.g. USD-SOFR-OIS)
        return "SOFR"
    if any(k in u for k in ("FEDERAL FUNDS", "FED FUND", "FED_FUNDS",
                            "H.15", "SERFF", "MIX23")) or u.startswith("USD-OIS"):
        return "FED_FUNDS"
    return s


def _resolve_tenor_dates(tenor_str, curve_id, as_of):
    """Resolve query tenor to exact (effective_date, maturity_date) pairs."""
    from Query.IRSwaps._CENTRAL_BANK_DATES import resolve_central_bank_tenor

    if tenor_str is None:
        return []

    t = str(tenor_str)
    parts = [p.strip() for p in t.split("/")]
    date_ranges = []

    for part in parts:
        if part.startswith("fomc_"):
            dates = resolve_central_bank_tenor(curve_id, part, as_of=as_of)
            if dates and len(dates) >= 2:
                date_ranges.append((
                    pd.Timestamp(dates[0]).date(),
                    pd.Timestamp(dates[1]).date(),
                ))

    return date_ranges


def dealer_flow_chart(
    query,
    date,
    start_hour=1,
    end_hour=17,
    n_jobs=8,
    show_tqdm=True,
    customer=False,
    match_tenor=True,
    scale_by_dv01=True,
    marker_at_traded_rate=False,
    rate_index=None,
):
    start = NY.localize(datetime.datetime(date.year, date.month, date.day, start_hour, 0))
    end = NY.localize(datetime.datetime(date.year, date.month, date.day, end_hour, 0))

    curve_mdp = IRSwapsMDP(source="BARCHART_STIRF-RL")
    ts = TimeseriesBuilder()
    rate_df = ts.get_timeseries(
        start=start, end=end,
        queries=[query],
        freq="1min",
        n_jobs=n_jobs,
        routers={"IRS": IRSwapsTB(curve_mdp, show_tqdm=show_tqdm)},
    )
    if rate_df.empty:
        raise RuntimeError("No rate data returned")

    col = rate_df.columns[0]
    rate_series = rate_df[col].dropna()

    conn = psycopg2.connect(resolve_pg_url())

    tenor_str = getattr(query, 'tenor', None)
    date_ranges = _resolve_tenor_dates(tenor_str, "USD-OIS", date)

    if match_tenor and date_ranges:
        where_clauses = []
        for eff, mat in date_ranges:
            where_clauses.append(
                f"(l.effective_date::date = '{eff.isoformat()}' "
                f"AND l.expiration_date::date = '{mat.isoformat()}')"
            )
        date_filter = " OR ".join(where_clauses)

        directions = pd.read_sql(f"""
            SELECT DISTINCT ON (d.unit_key)
                   d.unit_key, d.execution_timestamp, d.dealer_direction,
                   d.classification_method, d.direction_confidence,
                   d.structure_dv01, d.notional, d.fixed_rate, d.curve_mid,
                   d.spread_to_mid_bps, d.dealer_charge_bps,
                   d.rate_index_clean, d.trade_type, d.is_off_market,
                   d.tenor_query,
                   l.effective_date, l.expiration_date, l.tenor_label,
                   l.special_tenor_type, l.fomc_meeting_label
            FROM arbs_stir_direction_v1 d
            JOIN arbs_usd_swap_tape_legs_v2 l
              ON d.trade_id = l.trade_id
              AND l.as_of_date = d.as_of_date
            WHERE d.execution_timestamp::date = '{date.isoformat()}'
              AND d.dealer_direction IN ('PAID', 'RECEIVED')
              AND ({date_filter})
            ORDER BY d.unit_key, d.execution_timestamp
        """, conn)
        label_parts = [f"{e.isoformat()} -> {m.isoformat()}" for e, m in date_ranges]
        print(f"Exact tenor match ({', '.join(label_parts)}): {len(directions)} trades")
    else:
        directions = pd.read_sql(f"""
            SELECT DISTINCT ON (d.unit_key)
                   d.unit_key, d.execution_timestamp, d.dealer_direction,
                   d.classification_method, d.direction_confidence,
                   d.structure_dv01, d.notional, d.fixed_rate, d.curve_mid,
                   d.spread_to_mid_bps, d.dealer_charge_bps,
                   d.rate_index_clean, d.trade_type, d.is_off_market,
                   d.tenor_query,
                   l.effective_date, l.expiration_date, l.tenor_label,
                   l.special_tenor_type, l.fomc_meeting_label
            FROM arbs_stir_direction_v1 d
            LEFT JOIN arbs_usd_swap_tape_legs_v2 l
              ON d.trade_id = l.trade_id
              AND l.as_of_date = d.as_of_date
            WHERE d.execution_timestamp::date = '{date.isoformat()}'
              AND d.dealer_direction IN ('PAID', 'RECEIVED')
            ORDER BY d.unit_key, d.execution_timestamp
        """, conn)
        print(f"No tenor filter -- showing all {len(directions)} trades")

    conn.close()

    # Restrict to the rate index of the displayed curve. SOFR and FED_FUNDS
    # OIS forwards differ by the SOFR-FF basis (~2bp), so a correctly
    # classified SOFR print plotted against a FED_FUNDS outright line lands
    # on the wrong side of it and looks mislabeled. Default: infer the index
    # from the query curve via config.CURVE_FOR (pass an explicit rate_index
    # to override; None + an unmapped curve keeps all indices).
    # Infer the index from the query curve when not given, then canonicalise
    # so any Fed Funds convention (OIS Compound / H.15-OIS-Compound / H.15 /
    # SERFFX-MIX23) collapses to the same FED_FUNDS token before comparison.
    if rate_index is None:
        rate_index = _canonical_index(getattr(query, "curve", None))
    if rate_index is not None:
        rate_index = _canonical_index(rate_index)
        _n0 = len(directions)
        directions = directions[
            directions["rate_index_clean"].map(_canonical_index) == rate_index].copy()
        _drop = _n0 - len(directions)
        if _drop:
            print(f"rate_index filter = {rate_index}: kept "
                  f"{len(directions)}, dropped {_drop} off-index trade(s)")

    directions["execution_timestamp"] = pd.to_datetime(directions["execution_timestamp"])
    if rate_series.index.tz is not None:
        if directions["execution_timestamp"].dt.tz is None:
            directions["execution_timestamp"] = directions["execution_timestamp"].dt.tz_localize("UTC")
        directions["execution_timestamp"] = directions["execution_timestamp"].dt.tz_convert(rate_series.index.tz)
    mask = (directions["execution_timestamp"] >= rate_series.index.min()) & \
           (directions["execution_timestamp"] <= rate_series.index.max())
    directions = directions[mask]

    if customer:
        directions["direction"] = directions["dealer_direction"].map(_FLIP)
    else:
        directions["direction"] = directions["dealer_direction"]

    perspective = "customer" if customer else "dealer"

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=rate_series.index, y=rate_series.values,
        mode="lines", line=dict(width=2, color="#898781"),
        name=str(col), hovertemplate="%{y:.4f}<extra>rate</extra>",
    ))

    # Split on-market vs off-market for different marker shapes
    directions["_is_off_market"] = directions["is_off_market"].fillna(False).astype(bool)

    for direction, base_symbol, color in [
        ("RECEIVED", "triangle-up", "#008300"),
        ("PAID", "triangle-down", "#e34948"),
    ]:
        dir_sub = directions[directions["direction"] == direction]
        if dir_sub.empty:
            continue

        # On-market: triangle markers. Off-market: diamond markers.
        for off_market, marker_symbol, suffix in [
            (False, base_symbol, ""),
            (True, "diamond", " (off-mkt)"),
        ]:
            sub = dir_sub[dir_sub["_is_off_market"] == off_market]
            if sub.empty:
                continue

            if marker_at_traded_rate:
                y_vals = (sub["fixed_rate"].astype(float) * 100).values
            else:
                y_vals = []
                for t in sub["execution_timestamp"]:
                    idx = rate_series.index.get_indexer([t], method="nearest")
                    y_vals.append(float(rate_series.iloc[idx[0]]) if idx[0] != -1 else np.nan)

            if scale_by_dv01:
                sizes = np.clip(sub["structure_dv01"].fillna(0).values / 5000, 4, 25)
            else:
                sizes = 10

            eff_str = pd.to_datetime(sub["effective_date"]).dt.strftime("%Y-%m-%d").fillna("?").values
            exp_str = pd.to_datetime(sub["expiration_date"]).dt.strftime("%Y-%m-%d").fillna("?").values

            fig.add_trace(go.Scatter(
                x=sub["execution_timestamp"], y=y_vals,
                mode="markers",
                marker=dict(symbol=marker_symbol, size=sizes, color=color,
                            line=dict(width=1, color="white")),
                name=f"{direction}{suffix}",
                customdata=list(zip(
                    sub["structure_dv01"].fillna(0).round(0).values,
                    sub["notional"].fillna(0).apply(lambda x: f"{x/1e6:.1f}M").values,
                    sub["fixed_rate"].apply(lambda x: f"{x*100:.3f}%" if pd.notna(x) else "?").values,
                    sub["curve_mid"].apply(lambda x: f"{x*100:.3f}%" if pd.notna(x) else "?").values,
                    sub["spread_to_mid_bps"].apply(lambda x: f"{x:+.1f}bp" if pd.notna(x) else "?").values,
                    eff_str, exp_str,
                    sub["tenor_label"].fillna("?").values,
                    sub["direction_confidence"].fillna("?").values,
                    sub["classification_method"].fillna("?").values,
                    sub["trade_type"].fillna("?").values,
                    sub["rate_index_clean"].fillna("?").values,
                )),
                hovertemplate=(
                    f"<b>{perspective} {direction}{'  OFF-MARKET' if off_market else ''}</b><br>"
                    "DV01: %{customdata[0]:,.0f}  notional: %{customdata[1]}<br>"
                    "rate: %{customdata[2]}  mid: %{customdata[3]}  s2m: %{customdata[4]}<br>"
                    "eff: %{customdata[5]}  exp: %{customdata[6]}  tenor: %{customdata[7]}<br>"
                    "type: %{customdata[10]}  index: %{customdata[11]}<br>"
                    "conf: %{customdata[8]}  method: %{customdata[9]}"
                    "<extra></extra>"
                ),
            ))

    tenor_label = getattr(query, 'tenor', str(col))
    fig.update_layout(
        title=f"{tenor_label}{f' [{rate_index}]' if rate_index else ''} -- {date.isoformat()} {perspective} flow",
        yaxis_title="rate (bps)" if rate_series.max() < 1 else "rate (%)",
        xaxis_title="",
        height=500,
        plot_bgcolor="#fcfcfb",
        paper_bgcolor="white",
        font=dict(family="system-ui, -apple-system, sans-serif", color="#0b0b0b"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        hovermode="x unified",
        yaxis=dict(gridcolor="#e1e0d9", gridwidth=1),
        xaxis=dict(gridcolor="#e1e0d9", gridwidth=1),
    )
    return fig

In [21]:
q = UnifiedQuery(
    curve="USD-OIS-Q12xM12STIRT-SERFFX-MIX23",
    tenor="fomc_jul26",
    value=UnifiedValue.IRS_RATE,
)

fig = dealer_flow_chart(q, datetime.date(2026, 7, 2), customer=True, start_hour=1, end_hour=17, scale_by_dv01=False, marker_at_traded_rate=True, rate_index='FED_FUNDS')
fig.show()

Exact tenor match (2026-07-29 -> 2026-09-16): 75 trades
rate_index filter = FED_FUNDS: kept 61, dropped 14 off-index trade(s)
